# Notebook 04: Supervised Salary Prediction (Regression)
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Predict monthly salary in Million VND (`salary_mean_vnd`) using multi-modal features.
2. 80/10/10 split with strict zero leakage (preprocessors fit strictly on train split).
3. Evaluate 6 regression algorithms on full configuration E4 (Dummy, Linear, Ridge, RF, ExtraTrees, XGBoost).
4. Reconciled Ablation Study across E1 (Structured), E2 (+Year), E3 (+SBERT), and E4 (Multi-modal).
5. Longitudinal / Temporal Validation (Train: <=2025, Test: 2026).

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.models.regression import run_regression_experiments

## 1. Load Regression Dataset and Embeddings

In [ ]:
df_reg = pd.read_csv(config.PROCESSED_DATA_DIR / "regression_data.csv")
sbert_all = np.load(config.EMBEDDINGS_DIR / "sbert_embeddings.npy")
cleaned_df = pd.read_csv(config.PROCESSED_DATA_DIR / "cleaned_jobs.csv")

id_to_idx = {uid: idx for idx, uid in enumerate(cleaned_df['unique_job_id'])}
reg_indices = [id_to_idx[uid] for uid in df_reg['unique_job_id']]
reg_embeddings = sbert_all[reg_indices]

print(f"Regression samples: {df_reg.shape[0]}, SBERT embedding shape: {reg_embeddings.shape}")
print(f"Salary target (Million VND): Mean = {df_reg['salary_mean_vnd'].mean() / 1e6:.2f}M, Median = {df_reg['salary_mean_vnd'].median() / 1e6:.2f}M")

## 2. Train Models, Ablation Study, and Temporal Validation
This cell executes the actual training across all regression algorithms, runs ablation suite E1-E4, and validates on 2026 data.

In [ ]:
# Execute live training for salary regression
reg_results, ablation_results, temporal_results, best_model, meta, split_manifest = run_regression_experiments(
    df_reg,
    reg_embeddings
)

print("\n=== Full Model Comparison (Configuration E4 on Held-Out Test Set) ===")
display(reg_results)

print("\n=== Reconciled Ablation Study Summary (E1, E2, E3, E4) ===")
display(ablation_results)

if not temporal_results.empty:
    print("\n=== Temporal Validation (Train <=2025, Test 2026) ===")
    display(temporal_results)

## 3. Regression Diagnostic Visualizations

In [ ]:
for plot_name in [
    'regression_actual_vs_predicted.png',
    'regression_residuals_distribution.png',
    'regression_feature_importance.png',
    'regression_ablation_comparison.png'
]:
    p = config.REPORTS_DIR / 'figures' / plot_name
    if p.exists():
        print(f"\n--- {plot_name} ---")
        display(Image(filename=str(p)))